In [ ]:
import os
import random
import re
import shutil
from collections.abc import Callable, Sequence
from typing import Any, Literal

import cv2
import lightning as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import segmentation_models_pytorch as smp
import torch
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from lightning import LightningDataModule, LightningModule
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from torchmetrics import Accuracy, ConfusionMatrix, F1Score, MaxMetric, MeanMetric
from torchvision import tv_tensors
from torchvision.io import read_image
from torchvision.ops import masks_to_boxes
from torchvision.transforms import InterpolationMode
from torchvision.utils import save_image
from tqdm import tqdm

root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True)

from src.data.components.dataset import FetalBrainPlanesDataset
from src.data.components.transforms import PadToAspectRation, Resize
from src.data.utils.utils import show_numpy_images, show_pytorch_images
from src.models.head_segmentation_module import HeadSegmentationLitModule

data_dir = root / "data"
dataset_name = "FETAL_BRAIN_PLANES"
dataset_dir = data_dir / dataset_name

# Create FETAL_BRAIN_PLANES dataset

## Create Dataset directory

In [ ]:
if not dataset_dir.exists():
    dataset_dir.mkdir(parents=True)
    print(f"Dataset directory '{dataset_name}' created successfully.")
else:
    print(f"Dataset directory '{dataset_name}' already exists.")

## Prepare csv

### Load FETAL_PLANE csv

In [ ]:
fetal_planes = pd.read_csv(f"{data_dir}/FETAL_PLANES/data.csv")
fetal_planes = fetal_planes[["Image_name", "Patient_num", "Plane", "Brain_plane", "Train"]]
fetal_planes = fetal_planes.rename(columns={"Train ": "Train"})
fetal_planes = fetal_planes.reset_index(drop=True)
fetal_planes

### Assign Subset value (the same as for other experiments)

In [ ]:
train_fetal_planes = fetal_planes[fetal_planes["Train"] == 1].reset_index(drop=True)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=5724)
split = splitter.split(list(range(len(train_fetal_planes))), groups=train_fetal_planes["Patient_num"])
train_idx, val_idx = next(split)

fetal_planes["Subset"] = "test"

for idx, image_name in tqdm(
    enumerate(train_fetal_planes["Image_name"]), total=len(train_fetal_planes), desc="Add subset value"
):
    if idx in val_idx:
        subset = "val"
    else:
        subset = "train"
    fetal_planes.loc[fetal_planes["Image_name"] == image_name, "Subset"] = subset

fetal_planes

### Statistics

In [ ]:
fetal_planes["Subset"].value_counts()

In [ ]:
fetal_planes["Plane"].value_counts()

In [ ]:
fetal_planes["Brain_plane"].value_counts()

In [ ]:
fetal_planes[fetal_planes["Plane"] == "Fetal brain"]["Subset"].value_counts()

### Validate Subset on FetalBrainPlanesDataset

In [ ]:
train = FetalBrainPlanesDataset(
    data_dir=data_dir,
    subset="train",
)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=5724)
split = splitter.split(train, groups=train.img_labels["Patient_num"])
train_idx, val_idx = next(split)

train_df = train.img_labels.iloc[train_idx]
val_df = train.img_labels.iloc[val_idx]


incorrect = 0
for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    image_name = row["Image_name"]
    subset = row["Subset"]
    subset_train = train_df[train_df["Image_name"] == image_name]
    subset_val = val_df[val_df["Image_name"] == image_name]

    if subset == "train" and len(subset_train) != 1:
        incorrect += 1
    elif subset == "val" and len(subset_val) != 1:
        incorrect += 1

print(f"incorrect: {incorrect}")

### Validate Subset on fetal_head_segmentation

In [ ]:
fetal_head_segmentation = pd.read_csv(f"{data_dir}/FETAL_HEAD_SEGMENTATION/data.csv")
fetal_head_segmentation

incorrect = 0
for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes)):
    image_name = row["Image_name"]
    subset = row["Subset"]
    subset_2 = list(fetal_head_segmentation[fetal_head_segmentation["Image_name"] == image_name]["Subset"])
    if len(subset_2) == 1 and subset_2[0] != subset:
        incorrect += 1
        print(f"Error: {index} {row}")
        print(f"subset_2: {subset_2}")
        print()
        break

print(f"incorrect: {incorrect}")

### Save csv

In [ ]:
fetal_planes.to_csv(dataset_dir / "data.csv", index=False)
fetal_planes

##  Prepare Data

### Delete Dataset Image directory

In [ ]:
images_dir = dataset_dir / "Images"

if images_dir.exists():
    shutil.rmtree(images_dir)
    print(f"Directory '{images_dir}' and all its contents have been removed.")
else:
    print(f"Directory '{images_dir}' does not exist.")

### Create Dataset Images directory

In [ ]:
images_dir = dataset_dir / "Images"

if not images_dir.exists():
    images_dir.mkdir(parents=True)
    print(f"{images_dir.stem} directory created successfully.")
else:
    print(f"{images_dir.stem} directory already exists.")

### Process images

In [ ]:
fetal_planes = pd.read_csv(f"{dataset_dir}/data.csv")
fetal_planes

In [ ]:
# checkpoint_file = root / "logs" / "head_segmentation_train" / "runs" / "2025-08-25_13-36-03"
checkpoint_file = root / "logs" / "head_segmentation_train" / "runs" / "2025-08-26_17-11-43"
checkpoint = sorted(checkpoint_file.glob("checkpoints/epoch_*.ckpt"))[-1]
model = HeadSegmentationLitModule.load_from_checkpoint(str(checkpoint))
# disable randomness, dropout, etc...
model.eval()
model.to("cpu")
print("Loaded")

In [ ]:
def find_angle(mask):
    mask = mask.squeeze(0)

    # Get the indices of the non-zero elements
    coords = torch.nonzero(mask, as_tuple=False).float()
    # Center the data by subtracting the mean
    mean = torch.mean(coords, dim=0)
    centered_coords = coords - mean

    # Calculate the covariance matrix
    covariance_matrix = torch.matmul(centered_coords.T, centered_coords)
    # Perform eigenvalue decomposition to find eigenvectors (principal components)
    eigenvalues, eigenvectors = torch.linalg.eigh(covariance_matrix)
    # The eigenvector corresponding to the largest eigenvalue (the first principal component)
    principal_axis = eigenvectors[:, 1]
    # Calculate the angle of the principal axis
    angle = torch.atan2(principal_axis[0], principal_axis[1]) * 180 / torch.pi

    return angle.round().int()


def crop(image, x1, y1, x2, y2, pad=10):

    pad_x = int((x2 - x1) * (pad / 100.0))
    left_pad = pad_x // 2
    right_pad = pad_x - left_pad

    pad_y = int((y2 - y1) * (pad / 100.0))
    top_pad = pad_y // 2
    bottom_pad = pad_y - top_pad

    x1 = max(x1 - left_pad, 0)
    y1 = max(y1 - top_pad, 0)
    x2 = min(x2 + right_pad, image.shape[2])
    y2 = min(y2 + bottom_pad, image.shape[1])
    return TF.crop(image, y1, x1, y2 - y1, x2 - x1)


image_size = (192, 256)
transform = torch.nn.Sequential(
    T.Grayscale(),
    PadToAspectRation(image_size, fill=0),
    Resize(image_size, interpolation=T.InterpolationMode.BILINEAR),
    T.ToDtype(torch.float32, scale=True),
)
pad_to_aspect_ration = PadToAspectRation(image_size, fill=0)

fetal_planes = pd.read_csv(dataset_dir / "data.csv")

brain_images = []
unidentified_images = []
for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes), desc="Porcess images"):
    plane = row["Plane"]
    image_name = row["Image_name"]
    image_path = data_dir / "FETAL_PLANES" / "Images" / f"{image_name}.png"
    image = read_image(image_path)
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    elif image.shape[0] == 4:
        image = image[:3, :, :]

    if plane == "Fetal brain":
        with torch.no_grad():
            brain_plane = transform(image)
            logits = model(brain_plane.unsqueeze(0)).squeeze(0)
            prediction = (torch.sigmoid(logits) > 0.5).int()

            ones_counts = prediction.sum()
            total_pixels = prediction.numel()
            ones_percent = ones_counts.float() / total_pixels
            prediction_label = (ones_percent >= 0.05).int()

            if prediction_label != 1:
                unidentified_images.append(index)

            # if prediction_label == 1:
            # brain_plane = pad_to_aspect_ration(image)

            # angle = find_angle(prediction)
            # prediction = TF.resize(prediction, size=brain_plane.shape[1:], interpolation=T.InterpolationMode.NEAREST)
            # prediction = TF.rotate(prediction, angle=angle, interpolation=T.InterpolationMode.NEAREST)
            # brain_plane = TF.rotate(brain_plane, angle=angle, interpolation=T.InterpolationMode.BILINEAR)

            # x1, y1, x2, y2 = masks_to_boxes(prediction)[0].int()
            # image = crop(brain_plane, x1, y1, x2, y2, pad=5)

            # brain_images.append(index)
            # fetal_planes.loc[index, 'Identified'] = 1
            # else:
            #     unidentified_images.append(index)
            # fetal_planes.loc[index, 'Identified'] = 0
    # else:
    # fetal_planes.loc[index, 'Identified'] = 1

    # output_path = dataset_dir / "Images" / f"{image_name}.png"
    # image = image.permute(1, 2, 0).numpy()
    # image = Image.fromarray(image)
    # image.save(output_path)

fetal_planes["Identified"] = fetal_planes["Identified"].astype(int)
print(f"Number of unidentified images {len(unidentified_images)}")

In [ ]:
fetal_planes[fetal_planes["Plane"] == "Fetal brain"]["Brain_plane"].value_counts()

In [ ]:
fetal_planes[fetal_planes["Identified"] == 1]["Brain_plane"].value_counts()

In [ ]:
def print_subset_brain_planes(df, subset):
    print(subset)
    df = df[df["Identified"] == 1]
    df = df[df["Subset"] == subset]
    print(df["Brain_plane"].value_counts())
    print()


print_subset_brain_planes(fetal_planes, "train")
print_subset_brain_planes(fetal_planes, "val")
print_subset_brain_planes(fetal_planes, "test")

In [ ]:
fetal_planes.to_csv(dataset_dir / "data.csv", index=False)
fetal_planes

In [ ]:
random.shuffle(brain_images)
images = []
for i in brain_images[:64]:
    image_name = fetal_planes.Image_name[i]
    image_path = dataset_dir / "Images" / f"{image_name}.png"
    image = read_image(image_path)
    image = TF.to_grayscale(image)
    # image = TF.resize(image, (60, 80))
    brain_plane = fetal_planes.Brain_plane[i]
    subset = fetal_planes.Subset[i]
    images.append((image, f"{brain_plane} - {subset}"))

show_pytorch_images(images).show()

In [ ]:
images = []
for i in unidentified_images:
    image_name = fetal_planes.Image_name[i]
    image_path = data_dir / "FETAL_PLANES" / "Images" / f"{image_name}.png"
    image = read_image(image_path)
    image = TF.to_grayscale(image)
    # image = TF.resize(image, (60, 80))
    brain_plane = fetal_planes.Brain_plane[i]
    subset = fetal_planes.Subset[i]
    images.append((image, f"{brain_plane} - {subset}"))

show_pytorch_images(images).show()

In [ ]:
def extend_mean_median_plot(ax, mean, median, title):
    ax.axvline(mean, color="r", linestyle="dashed", linewidth=2)
    ax.axvline(median, color="b", linestyle="dashed", linewidth=2)
    ax.legend([f"mean {mean}", f"median {median}"], loc="upper left")
    ax.set_title(title, fontsize=14)


def print_resolution(scale, width):
    height = width / scale
    print(f"{height:.2f} / {width}")


data = []
for index, row in tqdm(fetal_planes.iterrows(), total=len(fetal_planes), desc="Read images shape"):
    image_name = row["Image_name"]
    image_path = dataset_dir / "Images" / f"{image_name}.png"
    image = read_image(image_path)
    data.append((image.shape[0], image.shape[1], image.shape[2]))

df_shape = pd.DataFrame(data=data)
mean = df_shape.mean()
median = df_shape.median()

axs = df_shape.hist(column=[1, 2], bins=100, figsize=(20, 5))
extend_mean_median_plot(axs[0][0], mean[1], median[1], "Height")
extend_mean_median_plot(axs[0][1], mean[2], median[2], "Width")

scale = median[2] / median[1]
print_resolution(scale, 64)  # 64 / 64
print_resolution(scale, 128)  # 96 / 128
print_resolution(scale, 256)  # 192 / 256